# Train the skin-lesion classifier (3 classes)

Run this on the PC with the RTX 4060 (WSL, `venv` kernel, `pip install -r training/requirements.txt`).
Set `DATA_DIR` in Cell 2 if the dataset is somewhere else. It produces the four files the kiosk reads:

| file | what | made by |
|---|---|---|
| `models/skin_classifier.tflite` | the model | Cell 11 |
| `models/labels.txt` | `benign`, `pre_cancerous`, `malignant` | Cell 11 |
| `models/thresholds.json` | screening threshold: 90% of cancers flagged on validation | Cell 12 (`training/calibrate.py`) |
| `models/temperature.json` | T for honest displayed confidence | Cell 12 (`training/calibrate.py`) |

What changed from the first version, and why:

1. **Split by lesion, not by image.** HAM10000 has several photos of many lesions. Splitting by image put
   the same lesion in train and test (40% of test images). `GroupShuffleSplit` on `lesion_id` fixes it;
   `training/splits.csv` records the split — commit it with the model, the paper needs it.
2. **Camera-style augmentation, per image.** The kiosk is a bare camera module, and the students demo by
   pointing it at a phone screen. `camera_augment` adds blur, colour cast, exposure drift, glare, a faint
   screen grid, low resolution and JPEG re-compression, so the model has seen that kind of photo.
3. **The threshold is fitted on the exported TFLite model with 4-view TTA** — exactly what the kiosk runs —
   not on the Keras model. The 90% sensitivity figure therefore describes the device.

When it finishes: copy `models/` (the four files) and `training/splits.csv` into the branch and commit.

In [ ]:
# Cell 0 — WSL GPU + all imports (do not split TensorFlow into another cell).
# TF loads CUDA on first import; if it imported earlier without LD_LIBRARY_PATH, GPU stays [] until Kernel Restart.
import os
import sys
from pathlib import Path


def _detect_venv() -> str:
    v = os.environ.get("VIRTUAL_ENV")
    if v:
        return v
    exe = Path(sys.executable).resolve()
    if exe.parent.name == "bin" and (exe.parent.parent / "pyvenv.cfg").is_file():
        return str(exe.parent.parent)
    raise RuntimeError(
        "Select kernel: skin-cancer-project/venv/bin/python (venv). "
        "VIRTUAL_ENV unset and interpreter is not inside a venv."
    )


venv = _detect_venv()
_pyv = f"{sys.version_info.major}.{sys.version_info.minor}"
_base = f"{venv}/lib/python{_pyv}/site-packages/nvidia"
_cuda_libs = [
    "/usr/lib/wsl/lib",
    f"{_base}/cuda_runtime/lib",
    f"{_base}/cuda_nvrtc/lib",
    f"{_base}/cublas/lib",
    f"{_base}/cudnn/lib",
    f"{_base}/cufft/lib",
    f"{_base}/curand/lib",
    f"{_base}/cusolver/lib",
    f"{_base}/cusparse/lib",
    f"{_base}/nccl/lib",
    f"{_base}/nvjitlink/lib",
]
_prev = os.environ.get("LD_LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = ":".join([p for p in _cuda_libs + ([_prev] if _prev else []) if p])

missing = [p for p in _cuda_libs if not Path(p).is_dir()]
if missing:
    print("WARNING: missing NVIDIA wheel dirs:", missing)

print("venv:", venv)
print("Python:", sys.version.split()[0])
print("LD_LIBRARY_PATH prefix:", os.environ["LD_LIBRARY_PATH"][:140] + "...")

# Help the dynamic linker load CUDA deps before TensorFlow's GPU plugin dlopens them
# (needed on some WSL + Jupyter kernels even when LD_LIBRARY_PATH is set).
import ctypes

_preloads = [
    Path(_base) / "cuda_runtime" / "lib" / "libcudart.so.12",
    Path(_base) / "cudnn" / "lib" / "libcudnn.so.9",
    Path(_base) / "cublas" / "lib" / "libcublas.so.12",
    Path(_base) / "cublas" / "lib" / "libcublasLt.so.12",
    Path(_base) / "cusolver" / "lib" / "libcusolver.so.11",
    Path(_base) / "cusparse" / "lib" / "libcusparse.so.12",
]
for _so in _preloads:
    if _so.is_file():
        try:
            ctypes.CDLL(str(_so), mode=ctypes.RTLD_GLOBAL)
            print("Preloaded:", _so.name)
        except OSError as _e:
            print("Preload skipped/failed:", _so, _e)

# Import TensorFlow in THIS cell immediately after LD_LIBRARY_PATH + preload (required on WSL).
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print("TensorFlow:", tf.__version__)
print("GPU devices:", gpus)

In [ ]:
# Cell 1 — Reproducibility + extra imports for the sensitivity-first pipeline.
# (Kept separate from the CUDA/TF import cell above on purpose.)
import json
import math
import random

import keras
from sklearn.metrics import (
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score,
    average_precision_score,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# Class index constants (kept in sync with LABELS defined in the class-collapse cell).
BENIGN_IDX, PRECANCER_IDX, MALIGNANT_IDX = 0, 1, 2

print("Seeds set to", SEED, "| keras", keras.__version__)


In [ ]:
# Cell 2: Paths + load metadata
# Where HAM10000 lives on this machine (not in git). Edit if yours is elsewhere.
DATA_DIR = "/home/hansdaduya/projects/skin-cancer-project/datasets/ham10000"
METADATA = os.path.join(DATA_DIR, "HAM10000_metadata.csv")
# The repo root, whether the notebook was opened from the root or from training/.
PROJECT = Path.cwd() if (Path.cwd() / "models").is_dir() else Path.cwd().parent
assert (PROJECT / "dermascan").is_dir(), f"Run this from the repo, not {Path.cwd()}"

df = pd.read_csv(METADATA)
print("Rows:", len(df), "| lesions:", df["lesion_id"].nunique())
print(df["dx"].value_counts())

In [ ]:
# Cell 3: Map image_id -> file path (handles both uppercase/lowercase folders)
candidate_dirs = [
    "HAM10000_images_part_1", "HAM10000_images_part_2",
    "ham10000_images_part_1", "ham10000_images_part_2",
]

image_paths = {}
for folder in candidate_dirs:
    folder_path = os.path.join(DATA_DIR, folder)
    if not os.path.isdir(folder_path):
        continue
    for fn in os.listdir(folder_path):
        if fn.endswith(".jpg"):
            image_paths[fn.replace(".jpg", "")] = os.path.join(folder_path, fn)

df["path"] = df["image_id"].map(image_paths)
print("Missing paths:", df["path"].isna().sum())
assert df["path"].isna().sum() == 0, "Some image paths are missing!"

In [ ]:
# Cell 4: Collapse 7 classes -> 3 classes
CLASS_MAP = {
    "mel": "malignant",
    "bcc": "malignant",
    "akiec": "pre_cancerous",
    "nv": "benign",
    "bkl": "benign",
    "df": "benign",
    "vasc": "benign",
}

LABELS = ["benign", "pre_cancerous", "malignant"]
label_to_idx = {l: i for i, l in enumerate(LABELS)}

df["label_3"] = df["dx"].map(CLASS_MAP)
df["label_idx"] = df["label_3"].map(label_to_idx)

print(df["label_3"].value_counts())

In [ ]:
# Cell 5: Split by LESION, not by image.
# HAM10000 photographs the same lesion more than once (lesion_id). An image-level split put
# 40% of test images' lesions into train, so the old test numbers were inflated. GroupShuffleSplit
# keeps every photo of a lesion on one side of the line. Stratification is approximate (groups
# cannot be stratified exactly); the printed per-class counts show it is close enough.
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, temp_idx = next(gss.split(df, df["label_idx"], groups=df["lesion_id"]))
train_df, temp_df = df.iloc[train_idx].reset_index(drop=True), df.iloc[temp_idx].reset_index(drop=True)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_idx, test_idx = next(gss2.split(temp_df, temp_df["label_idx"], groups=temp_df["lesion_id"]))
val_df, test_df = temp_df.iloc[val_idx].reset_index(drop=True), temp_df.iloc[test_idx].reset_index(drop=True)

# Prove there is no leakage: no lesion appears in two splits.
assert not set(train_df.lesion_id) & set(val_df.lesion_id)
assert not set(train_df.lesion_id) & set(test_df.lesion_id)
assert not set(val_df.lesion_id) & set(test_df.lesion_id)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
for name, frame in (("train", train_df), ("val", val_df), ("test", test_df)):
    print(name, frame["label_3"].value_counts().to_dict())

# Record the split so eval.py and the paper use exactly these images.
splits = pd.concat([train_df.assign(split="train"), val_df.assign(split="val"), test_df.assign(split="test")])
splits[["split", "lesion_id", "image_id", "dx", "label_3", "label_idx", "path"]].to_csv(PROJECT / "training" / "splits.csv", index=False)

class_weights = compute_class_weight(class_weight="balanced", classes=np.arange(len(LABELS)), y=train_df["label_idx"].values)
print("Reference class weights (not used; batches are resampled instead):", {i: round(float(w), 2) for i, w in enumerate(class_weights)})

train_df_by_class = {idx: train_df[train_df["label_idx"] == idx].reset_index(drop=True) for idx in range(len(LABELS))}

In [ ]:
# Cell 6: tf.data pipeline — balanced resampling + augmentation
# Images stay float32 in [0, 255] end to end: EfficientNet rescales internally, and the kiosk
# (dermascan/classifier.py) feeds the TFLite model raw [0, 255] too. Never add /255 here.
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Sample the three classes equally: the rare classes are oversampled (and augmented) instead of
# relying on a 10x loss weight. Shift toward benign here if benign recall collapses.
RESAMPLE_WEIGHTS = [1 / 3, 1 / 3, 1 / 3]
STEPS_PER_EPOCH = len(train_df) // BATCH_SIZE


def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return tf.cast(img, tf.float32), label  # [0, 255]


# Geometric + mild photometric augmentation, per batch (Keras samples per image inside it).
# Colour jitter stays mild here because colour is diagnostic for melanoma.
augmenter = keras.Sequential(
    [
        keras.layers.RandomFlip("horizontal_and_vertical", seed=SEED),
        keras.layers.RandomRotation(0.2, fill_mode="reflect", seed=SEED),
        keras.layers.RandomZoom(0.15, fill_mode="reflect", seed=SEED),
        keras.layers.RandomTranslation(0.1, 0.1, fill_mode="reflect", seed=SEED),
        keras.layers.RandomContrast(0.2, seed=SEED),
        keras.layers.RandomBrightness(0.15, value_range=(0.0, 255.0), seed=SEED),
    ],
    name="augmenter",
)


# --- Camera-style augmentation, one image at a time --------------------------------------
# The kiosk is a fixed-focus camera module under room light, often pointed at a phone or
# laptop screen. Each helper imitates one thing that camera does to a picture. 60% of images
# get the treatment; within it each effect is a coin flip, so every image is different.
# All helpers take and return one float image in [0, 1], shape (224, 224, 3).
CAMERA_AUG_PROB = 0.6


def _coin(p):
    return tf.random.uniform([]) < p


def _gaussian_blur(x, sigma):
    """Separable Gaussian blur, 9-tap, sigma chosen per image.

    The edges are padded by mirroring, not with zeros: zero padding darkens a 4 px frame
    around the image, and the rotation augmentation then mirrors that frame into lines.
    """
    r = tf.range(-4.0, 5.0)
    k = tf.exp(-(r ** 2) / (2.0 * sigma ** 2))
    k = k / tf.reduce_sum(k)
    kx = tf.tile(tf.reshape(k, [1, 9, 1, 1]), [1, 1, 3, 1])
    ky = tf.tile(tf.reshape(k, [9, 1, 1, 1]), [1, 1, 3, 1])
    y = tf.pad(x[None], [[0, 0], [4, 4], [4, 4], [0, 0]], mode="REFLECT")
    y = tf.nn.depthwise_conv2d(y, kx, [1, 1, 1, 1], "VALID")
    y = tf.nn.depthwise_conv2d(y, ky, [1, 1, 1, 1], "VALID")
    return y[0]


def _defocus(x):
    """Fixed-focus softness: Gaussian blur, sigma 0.5-2 px."""
    return _gaussian_blur(x, tf.random.uniform([], 0.5, 2.0))


def _colour(x):
    """White balance, saturation and exposure drift."""
    x = tf.image.random_hue(x, 0.025)  # wider turns skin green
    x = tf.image.random_saturation(x, 0.7, 1.3)
    return tf.pow(tf.clip_by_value(x, 1e-4, 1.0), tf.random.uniform([], 0.75, 1.35))


def _glare(x):
    """A soft bright blob: a reflection on a screen or on shiny skin."""
    cx, cy = tf.random.uniform([], 0.2, 0.8), tf.random.uniform([], 0.2, 0.8)
    r = tf.random.uniform([], 0.15, 0.4)
    yy, xx = tf.meshgrid(tf.linspace(0.0, 1.0, IMG_SIZE), tf.linspace(0.0, 1.0, IMG_SIZE), indexing="ij")
    blob = tf.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2.0 * r * r))
    return x + tf.random.uniform([], 0.15, 0.4) * blob[:, :, None]


def _screen(x):
    """A photo of a screen: faint pixel grid (3-8 px pitch), softened by 1 px, slightly blue."""
    pitch = tf.random.uniform([], 3, 9, dtype=tf.int32)
    line = tf.cast(tf.math.floormod(tf.range(IMG_SIZE), pitch) == 0, tf.float32)
    grid = 1.0 - tf.random.uniform([], 0.06, 0.18) * tf.maximum(line[:, None], line[None, :])
    x = _gaussian_blur(x * grid[:, :, None], 1.0)
    a = tf.random.uniform([], 0.02, 0.08)
    return x * tf.stack([1.0 - a, 1.0, 1.0 + a])  # R down, B up


def _low_res(x):
    """A small or distant subject: down to 160 px and back up."""
    return tf.image.resize(tf.image.resize(x, [160, 160]), [IMG_SIZE, IMG_SIZE])


def _jpeg(x):
    """JPEG re-compression, quality 40-90 (a photo of a screen is a JPEG of a JPEG)."""
    x8 = tf.image.convert_image_dtype(tf.clip_by_value(x, 0.0, 1.0), tf.uint8, saturate=True)
    x8 = tf.image.random_jpeg_quality(x8, 40, 90)
    return tf.image.convert_image_dtype(x8, tf.float32)


def camera_augment(img, label):
    x = tf.clip_by_value(img / 255.0, 0.0, 1.0)
    for p, fn in ((0.5, _defocus), (0.6, _colour), (0.3, _glare), (0.5, _screen), (0.3, _low_res), (0.5, _jpeg)):
        x = tf.cond(_coin(p), lambda x=x, fn=fn: fn(x), lambda x=x: x)
        x = tf.ensure_shape(tf.clip_by_value(x, 0.0, 1.0), [IMG_SIZE, IMG_SIZE, 3])
    return x * 255.0, label


def maybe_camera_augment(img, label):
    return tf.cond(_coin(CAMERA_AUG_PROB), lambda: camera_augment(img, label), lambda: (img, label))


def _augment_batch(images, labels):
    return augmenter(images, training=True), labels


def make_class_ds(frame):
    ds = tf.data.Dataset.from_tensor_slices((frame["path"].values, frame["label_idx"].values))
    ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True).repeat()
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    return ds.map(maybe_camera_augment, num_parallel_calls=AUTOTUNE)


def make_train_dataset():
    class_dss = [make_class_ds(train_df_by_class[i]) for i in range(len(LABELS))]
    ds = tf.data.Dataset.sample_from_datasets(class_dss, weights=RESAMPLE_WEIGHTS, seed=SEED)
    ds = ds.batch(BATCH_SIZE).map(_augment_batch, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


def make_eval_dataset(dataframe):
    ds = tf.data.Dataset.from_tensor_slices((dataframe["path"].values, dataframe["label_idx"].values))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = make_train_dataset()           # infinite — drive with STEPS_PER_EPOCH
val_ds = make_eval_dataset(val_df)        # clean: validation is real dermoscopy
test_ds = make_eval_dataset(test_df)
calib_ds = make_eval_dataset(train_df)    # finite, no augmentation -> TFLite quantisation sample

print("Steps per epoch:", STEPS_PER_EPOCH, "| batch spec:", train_ds.element_spec[0])

# Look before you train: 16 augmented training images.
_xb, _yb = next(iter(train_ds))
fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for ax, im, lb in zip(axes.ravel(), _xb.numpy()[:16], _yb.numpy()[:16]):
    ax.imshow(im.astype("uint8")); ax.set_title(LABELS[lb], fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Cell 6b: Sensitivity-oriented loss + validation metrics
# Focal loss focuses training on the hard, rare minority lesions instead of letting the
# 80%-benign majority dominate. With resampling already balancing the batches we use a mild
# alpha and gamma=2.0.
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = [0.25, 0.35, 0.40]  # benign, pre_cancerous, malignant (slight minority emphasis)


def make_sparse_focal_loss(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA):
    alpha_t = tf.constant(alpha, dtype=tf.float32) if alpha is not None else None

    def loss_fn(y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        p_t = tf.gather(y_pred, y_true, batch_dims=1)           # prob of the true class
        ce = -tf.math.log(p_t)
        modulating = tf.pow(1.0 - p_t, gamma)
        per_example = modulating * ce
        if alpha_t is not None:
            per_example = tf.gather(alpha_t, y_true) * per_example
        return tf.reduce_mean(per_example)

    return loss_fn


focal_loss = make_sparse_focal_loss()


class ValSensitivityMetrics(tf.keras.callbacks.Callback):
    """Logs macro-F1 + malignant/pre-cancer recall each epoch so EarlyStopping and
    ModelCheckpoint can monitor what matters for a screener instead of raw accuracy.

    NOTE: must be listed BEFORE EarlyStopping/ModelCheckpoint in the callbacks list so the
    extra keys exist in `logs` when those callbacks read them.
    """

    def __init__(self, val_dataset, labels):
        super().__init__()
        self.val_dataset = val_dataset
        self.labels = labels

    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        y_true, y_pred = [], []
        for xb, yb in self.val_dataset:
            probs = self.model.predict(xb, verbose=0)
            y_true.extend(yb.numpy().tolist())
            y_pred.extend(np.argmax(probs, axis=1).tolist())
        macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
        rec = recall_score(y_true, y_pred, average=None, labels=range(len(self.labels)), zero_division=0)
        cancer_true = [1 if t in (PRECANCER_IDX, MALIGNANT_IDX) else 0 for t in y_true]
        cancer_pred = [1 if p in (PRECANCER_IDX, MALIGNANT_IDX) else 0 for p in y_pred]
        screen_sens = recall_score(cancer_true, cancer_pred, zero_division=0)
        logs["val_macro_f1"] = float(macro_f1)
        logs["val_mal_recall"] = float(rec[MALIGNANT_IDX])
        logs["val_pre_recall"] = float(rec[PRECANCER_IDX])
        logs["val_screen_sens"] = float(screen_sens)
        print(
            f"  val_macro_f1={macro_f1:.3f} "
            f"mal_recall={rec[MALIGNANT_IDX]:.3f} pre_recall={rec[PRECANCER_IDX]:.3f} "
            f"screen_sens(cancer-vs-not)={screen_sens:.3f}"
        )


print("focal_loss + ValSensitivityMetrics ready")


In [ ]:
# Cell 7: Build model (EfficientNetB0) — sensitivity-first compile
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
)
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
outputs = Dense(len(LABELS), activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=focal_loss,                       # focal loss instead of plain cross-entropy
    metrics=["accuracy"],                  # macro-F1 / recalls come from ValSensitivityMetrics
)
model.summary()


In [ ]:
# Cell 8: Train phase 1 (frozen backbone, train the head)
# No class_weight now — the resampled train_ds already balances classes per batch.
# ValSensitivityMetrics is FIRST so val_macro_f1 exists when the other callbacks read logs.
callbacks_phase1 = [
    ValSensitivityMetrics(val_ds, LABELS),
    EarlyStopping(patience=6, restore_best_weights=True, monitor="val_macro_f1", mode="max"),
    ReduceLROnPlateau(patience=3, factor=0.5, monitor="val_loss"),
    ModelCheckpoint(
        str(PROJECT / "training" / "best_model_phase1.keras"),
        save_best_only=True, monitor="val_macro_f1", mode="max",
    ),
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    steps_per_epoch=STEPS_PER_EPOCH,
    callbacks=callbacks_phase1,
)


In [ ]:
# Cell 9: Fine-tune phase 2 — unfreeze the FULL backbone with a cosine LR
# The old version only unfroze the last 30 layers at lr=1e-5, so the backbone never adapted
# to dermoscopy (underfit). Unfreeze everything but keep BatchNorm layers frozen to preserve
# the ImageNet running statistics (standard, stabilizes EfficientNet fine-tuning).
base_model.trainable = True
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

FT_EPOCHS = 40
cosine_lr = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-4,
    decay_steps=STEPS_PER_EPOCH * FT_EPOCHS,
)
model.compile(
    optimizer=tf.keras.optimizers.Adam(cosine_lr),
    loss=focal_loss,
    metrics=["accuracy"],
)

# CosineDecay is a schedule (not a mutable LR), so no ReduceLROnPlateau here.
callbacks_phase2 = [
    ValSensitivityMetrics(val_ds, LABELS),
    EarlyStopping(patience=8, restore_best_weights=True, monitor="val_macro_f1", mode="max"),
    ModelCheckpoint(
        str(PROJECT / "training" / "best_model_phase2.keras"),
        save_best_only=True, monitor="val_macro_f1", mode="max",
    ),
]

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FT_EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    callbacks=callbacks_phase2,
)


In [ ]:
# Cell 10: Evaluate — screener-relevant metrics (sensitivity-first)
def collect_probs(dataset):
    yt, yp = [], []
    for xb, yb in dataset:
        probs = model.predict(xb, verbose=0)
        yt.extend(yb.numpy().tolist())
        yp.append(probs)
    return np.array(yt), np.concatenate(yp, axis=0)


# Keep val arrays around for threshold tuning + temperature scaling in the next cell.
y_val_true, y_val_prob = collect_probs(val_ds)
y_test_true, y_test_prob = collect_probs(test_ds)
y_test_pred = np.argmax(y_test_prob, axis=1)

print("=== Per-class report (argmax baseline) ===")
print(classification_report(y_test_true, y_test_pred, target_names=LABELS, digits=3))

# Per-class sensitivity (recall) and specificity from the confusion matrix.
cm = confusion_matrix(y_test_true, y_test_pred, labels=range(len(LABELS)))
total = cm.sum()
print("=== Per-class sensitivity / specificity ===")
for i, name in enumerate(LABELS):
    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp
    tn = total - tp - fn - fp
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    print(f"  {name:14s} sensitivity={sens:.3f}  specificity={spec:.3f}")

print(f"\nmacro-F1 = {f1_score(y_test_true, y_test_pred, average='macro'):.3f}")

# One-vs-rest ROC-AUC / PR-AUC per class (uses probabilities, threshold-independent).
print("=== ROC-AUC / PR-AUC (one-vs-rest) ===")
for i, name in enumerate(LABELS):
    yt_bin = (y_test_true == i).astype(int)
    if yt_bin.sum() == 0:
        continue
    roc = roc_auc_score(yt_bin, y_test_prob[:, i])
    pr = average_precision_score(yt_bin, y_test_prob[:, i])
    print(f"  {name:14s} ROC-AUC={roc:.3f}  PR-AUC={pr:.3f}")

# Collapsed "cancerous vs not" screening view — the number that matters for the use case.
p_cancer = y_test_prob[:, PRECANCER_IDX] + y_test_prob[:, MALIGNANT_IDX]
cancer_true = np.isin(y_test_true, [PRECANCER_IDX, MALIGNANT_IDX]).astype(int)
cancer_pred = (p_cancer >= 0.5).astype(int)
print("\n=== Screening view: cancerous (pre+mal) vs benign, threshold 0.5 ===")
print(classification_report(cancer_true, cancer_pred, target_names=["benign", "cancerous"], digits=3))
print(f"screening ROC-AUC = {roc_auc_score(cancer_true, p_cancer):.3f}")

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix (argmax)")
plt.show()


**The screening threshold and the temperature are not fitted here.** They used to be fitted on the
Keras model's single-view output, but the kiosk runs the *quantised TFLite* model with *4-view TTA*,
and those give slightly different probabilities — enough to move a threshold of ~0.1. They are now
fitted after export, on the exact device path, by `training/calibrate.py` (Cell 12).

In [ ]:
# Cell 11: Export .tflite + labels. (The Keras model stays in training/, which git ignores.)
MODELS = PROJECT / "models"
MODELS.mkdir(exist_ok=True)
model.save(PROJECT / "training" / "skin_classifier_full.keras")


def representative_dataset():
    # A clean sample of the training set, used to choose the int8 ranges for the activations.
    for x_batch, _ in calib_ds.take(100):
        yield [x_batch]


# Weights AND activations quantised to int8 where possible; input and output stay float32,
# which is what dermascan/classifier.py feeds and reads. Faster on the Pi's CPU than
# float or weight-only quantisation, and the same recipe as the model that shipped before.
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
tflite_model = converter.convert()
(MODELS / "skin_classifier.tflite").write_bytes(tflite_model)
(MODELS / "labels.txt").write_text("\n".join(LABELS) + "\n")
print(f"Saved models/skin_classifier.tflite ({len(tflite_model)} bytes) and models/labels.txt")

In [ ]:
# Cell 12: Fit threshold + temperature on the EXPORTED model with 4-view TTA, then score the test set.
# Both scripts use dermascan/classifier.py, the same code the kiosk runs.
import subprocess

for script in ("training/calibrate.py", "training/eval.py"):
    print(f"\n$ python {script}")
    subprocess.run([sys.executable, script], cwd=PROJECT, check=True)

print("\nDone. Commit models/ (four files) and training/splits.csv, and paste the table into docs/METRICS.md.")